# ClaimWise Insurance Prediction — 16. Anomaly Detection (Isolation Forest)

## 1. Objective

Flag the ClaimWise records that are **unusual in the feature space**, using Isolation Forest — an
unsupervised anomaly detector that isolates observations by randomly splitting feature values.

| Rule | How it is respected |
|---|---|
| target is not an input | `loss` is dropped before fitting (printed below) |
| no fraud claims | ClaimWise has **no fraud label**; an anomaly is *unusual*, never *fraudulent* |
| reproducibility | `random_state=42` |

Artifacts go to `Outputs/anomaly/`.

## 2. Import Libraries

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Walk upwards from the notebook folder until the ClaimWise project is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Dataset").is_dir() and (candidate / "Src").is_dir():
            return candidate
    raise FileNotFoundError(f"ClaimWise project root not found above: {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "Dataset"
SRC_DIR = PROJECT_ROOT / "Src"
MODELS_DIR = PROJECT_ROOT / "Models"
OUTPUTS_DIR = PROJECT_ROOT / "Outputs"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from academic_common import ensure_dirs, save_figure, style_axes

from academic_common import ANOMALY_DIR

ensure_dirs()
print("anomaly output folder:", ANOMALY_DIR)

anomaly output folder: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/anomaly


In [3]:
# Src/model_training_utils.py switches matplotlib to the non-interactive Agg backend so the
# command-line scripts can run without a display. Switch back to the inline backend here so
# that every figure below is rendered inside this notebook.
%matplotlib inline

## 3. Load the ClaimWise Features (target excluded)

In [4]:
# The real ClaimWise preprocessed data. The target "loss" is REMOVED: unsupervised methods
# must not see the target as an input feature.
df = pd.read_csv(DATASET_DIR / "05_preprocessed_final.csv")
target_present = "loss" in df.columns
features = df.drop(columns=["loss"]) if target_present else df
loss_only = df["loss"] if target_present else None

print("raw preprocessed shape:", df.shape)
print("features used:", features.shape, "| 'loss' excluded from the inputs:", target_present)
print("features numeric:", all(pd.api.types.is_numeric_dtype(d) for d in features.dtypes))
print("missing values:", int(features.isna().sum().sum()))
print("duplicate rows:", int(features.duplicated().sum()))

raw preprocessed shape: (50000, 131)
features used: (50000, 130) | 'loss' excluded from the inputs: True
features numeric: True
missing values: 0


duplicate rows: 0


## 4. Isolation Forest

`IsolationForest` builds random isolation trees; short isolation paths identify anomalies.
With `contamination="auto"` the decision threshold is taken from the model's own score distribution
(no percentage is invented by hand), and `n_estimators=200` keeps the result stable.

In [5]:
from sklearn.ensemble import IsolationForest

detector = IsolationForest(n_estimators=200, contamination="auto", random_state=42, n_jobs=-1)
detector.fit(features)

raw_predictions = detector.predict(features)          # 1 = normal, -1 = anomaly
scores = detector.decision_function(features)         # higher = more normal

anomaly_mask = raw_predictions == -1
n_total = len(features)
n_anomaly = int(anomaly_mask.sum())

print("records scored      :", n_total)
print("normal observations :", n_total - n_anomaly)
print("anomalous observations:", n_anomaly)
print("anomaly percentage  : {:.2f}%".format(100 * n_anomaly / n_total))
print("offset_ used by predict():", round(float(detector.offset_), 6))

records scored      : 50000
normal observations : 48359
anomalous observations: 1641
anomaly percentage  : 3.28%
offset_ used by predict(): -0.5


In [6]:
result = pd.DataFrame({
    "is_anomaly": anomaly_mask.astype(int),
    "anomaly_score": scores,
    "loss_context_only": loss_only.to_numpy(),   # never an input - shown only as context afterwards
})
summary = pd.DataFrame([{
    "records": n_total,
    "normal": n_total - n_anomaly,
    "anomalies": n_anomaly,
    "anomaly_%": round(100 * n_anomaly / n_total, 2),
    "mean_score_normal": round(float(scores[~anomaly_mask].mean()), 4),
    "mean_score_anomaly": round(float(scores[anomaly_mask].mean()), 4),
}])
display(summary)

summary_path = ANOMALY_DIR / "anomaly_summary.csv"
summary.to_csv(summary_path, index=False)
result.to_csv(ANOMALY_DIR / "anomaly_scores.csv", index=False)
print("Saved:", summary_path)
print("Saved:", ANOMALY_DIR / "anomaly_scores.csv")

,records,normal,anomalies,anomaly_%,mean_score_normal,mean_score_anomaly
0,50000,48359,1641,3.28,0.0754,-0.0243


Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/anomaly/anomaly_summary.csv
Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/anomaly/anomaly_scores.csv


## 5. Score Distribution

The decision function separates the two groups: anomalies have lower (more negative) scores.

In [7]:
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.hist(scores[~anomaly_mask], bins=60, color="#3F7D4A", alpha=0.75, label="normal")
ax.hist(scores[anomaly_mask], bins=60, color="#D4512A", alpha=0.85, label="anomaly")
ax.axvline(detector.offset_, color="#18181B", linestyle="--",
           label=f"decision threshold ({detector.offset_:.3f})")
style_axes(ax, "Isolation Forest anomaly scores", "decision function score", "records")
ax.legend()
path = save_figure(fig, ANOMALY_DIR / "anomaly_score_distribution.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/anomaly/anomaly_score_distribution.png


## 6. Where the Anomalies Sit (2D PCA view)

PCA is used **only to draw a picture**: the detector ran on all 130 scaled-free raw features; the
plot projects the labels onto two principal components.

In [8]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

projected = PCA(n_components=2, random_state=42).fit_transform(StandardScaler().fit_transform(features))
n_show = 15000
show_idx = np.random.RandomState(42).choice(len(projected), size=n_show, replace=False)

fig, ax = plt.subplots(figsize=(9, 6.4))
normal_show = show_idx[~anomaly_mask[show_idx]]
anom_show = show_idx[anomaly_mask[show_idx]]
ax.scatter(projected[normal_show, 0], projected[normal_show, 1],
           c="#3F7D4A", s=10, alpha=0.5, label=f"normal ({len(normal_show)} shown)")
ax.scatter(projected[anom_show, 0], projected[anom_show, 1],
           c="#D4512A", s=18, alpha=0.9, edgecolors="white", linewidths=0.3,
           label=f"anomaly ({len(anom_show)} shown)")
style_axes(ax, f"Anomalies in a 2D PCA projection ({n_show} of {n_total} rows)",
           "PC1", "PC2")
ax.legend(loc="best", fontsize=9)
path = save_figure(fig, ANOMALY_DIR / "anomaly_pca_visualization.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/anomaly/anomaly_pca_visualization.png


## 7. Context Check — Loss Distribution of Flagged Records

`loss` was **not** used by the detector. Reading it back afterwards is an analysis of whether the
flagged records also look unusual in the claim amount. This is context, not training, and it must
not be read as fraud evidence.

In [9]:
context = pd.DataFrame({
    "group": np.where(anomaly_mask, "anomaly", "normal"),
    "loss": loss_only.to_numpy(),
}).groupby("group")["loss"].agg(["count", "mean", "median", "min", "max"]).round(2)
display(context)

fig, ax = plt.subplots(figsize=(8, 5))
data = [pd.Series(loss_only.to_numpy()[~anomaly_mask]).sample(4000, random_state=42).to_numpy(),
        loss_only.to_numpy()[anomaly_mask]]
ax.boxplot(data, tick_labels=["normal", "anomaly"], patch_artist=True,
           boxprops=dict(facecolor="#FCFAF6"))
style_axes(ax, "loss of normal vs flagged records (context only — loss was not an input)",
           "", "loss")
path = save_figure(fig, ANOMALY_DIR / "anomaly_loss_context.png")
print("Saved:", path)

,count,mean,median,min,max
group,,,,,
anomaly,1641,7397.88,6235.66,402.8,85923.56
normal,48359,2886.96,2057.38,10.0,79623.52


Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/anomaly/anomaly_loss_context.png


## 8. Conclusion

Actual results of this executed run:

* Records scored: **50,000**
* Normal: **48,359** · Anomalous: **1,641**
* Anomaly percentage: **3.28%**
* Decision threshold taken from the model's own score distribution (`contamination="auto"`).

**An anomaly is an unusual observation in the feature space. It does not mean the claim is
fraudulent** — ClaimWise contains no verified fraud labels, so no such statement can be made.

Artifacts: `anomaly_summary.csv`, `anomaly_scores.csv`, `anomaly_score_distribution.png`,
`anomaly_pca_visualization.png`, `anomaly_loss_context.png`.